In [1]:
#Install required tool:
!pip install -q yt-dlp openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 15.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 57.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.7 MB/s eta 0:00:00


In [2]:
#Import libraries
import json
from pathlib import Path

import whisper
import yt_dlp

In [3]:
VIDEO_IDS: list[str] = [
    # t_01
    "fIRM0bEZTpc",

    # t_02 ... t_08
    "MBtHiyY3FPo",
    "7F_HjxCLo6o",
    "JjSvBhc1gcg",
    "Qro3G8lKl0Y",
    "_nsLed3AHy8",
    "85UTQ6OWDFU",
    "KrHmB8nCU3U",
]

In [4]:
#configuration
MODEL_NAME = "medium"   # tiny, base, small, medium, large
LANGUAGE = "es"         # usa None si quieres detección automática
TASK = "transcribe"     # o "translate"

BASE_DIR = Path("/content")
AUDIO_DIR = BASE_DIR / "audio"
RAW_DIR = BASE_DIR / "raw"

AUDIO_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

In [5]:
#auxiliary functions
def hhmmss(seconds: float) -> str:
    total = int(round(seconds))
    h = total // 3600
    m = (total % 3600) // 60
    s = total % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def find_downloaded_audio(video_id: str) -> Path | None:
    matches = list(AUDIO_DIR.glob(f"{video_id}.*"))
    return matches[0] if matches else None


def download_audio(video_id: str) -> Path:
    url = f"https://www.youtube.com/watch?v={video_id}"
    outtmpl = str(AUDIO_DIR / f"{video_id}.%(ext)s")

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": outtmpl,
        "quiet": True,
        "noplaylist": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    audio_path = find_downloaded_audio(video_id)
    if audio_path is None:
        raise FileNotFoundError(f"Audio file not found for {video_id}")

    return audio_path


def whisper_to_segments(result: dict) -> list[dict]:
    out = []

    for seg in result.get("segments", []):
        start = float(seg.get("start", 0.0))
        end = float(seg.get("end", start))
        text = str(seg.get("text", "")).replace("\n", " ").strip()

        if not text:
            continue

        out.append(
            {
                "text": text,
                "start": start,
                "duration": max(0.0, end - start),
                "start_hhmmss": hhmmss(start),
                "end_hhmmss": hhmmss(end),
            }
        )

    return out

In [6]:
#load whispers
model = whisper.load_model(MODEL_NAME)
print(f"Whisper model loaded: {MODEL_NAME}")

100%|█████████████████████████████████████| 1.42G/1.42G [00:15<00:00, 96.1MiB/s]


Whisper model loaded: medium


In [7]:
#single video processing
def process_video(video_id: str, index: int) -> dict:
    print(f"Processing {video_id}...")

    audio_path = download_audio(video_id)
    print(f"  Audio: {audio_path.name}")

    result = model.transcribe(
        str(audio_path),
        language=LANGUAGE,
        task=TASK,
        verbose=False,
    )

    segments = whisper_to_segments(result)

    payload = {
        "t_id": f"t_{index:02d}",
        "video_id": video_id,
        "language": result.get("language", LANGUAGE or "unknown"),
        "segments": segments,
    }

    out_file = RAW_DIR / f"t_{index:02d}.json"
    out_file.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"  Saved: {out_file.name} | segments: {len(segments)}")
    return payload

In [8]:
#massive_video processing & transcription fecthing
all_payloads = []
errors = []

for i, video_id in enumerate(VIDEO_IDS, start=1):
    print(f"\n[{i}/{len(VIDEO_IDS)}] {video_id}")
    try:
        payload = process_video(video_id, i)
        all_payloads.append(payload)
    except Exception as e:
        print(f"  ERROR: {type(e).__name__}: {e}")
        errors.append({"video_id": video_id, "error": f"{type(e).__name__}: {e}"})

merged = {
    "total_videos": len(all_payloads),
    "videos": all_payloads,
    "errors": errors,
}

merged_path = RAW_DIR / "all_transcripts.json"
merged_path.write_text(
    json.dumps(merged, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nDone.")
print(f"OK -> {merged_path}")
print(f"Success: {len(all_payloads)}/{len(VIDEO_IDS)}")
if errors:
    print("Errors:")
    for err in errors:
        print("-", err)


[1/8] fIRM0bEZTpc
Processing fIRM0bEZTpc...


  Audio: fIRM0bEZTpc.webm


 99%|█████████▉| 320812/324688 [05:55<00:04, 903.62frames/s]


  Saved: t_01.json | segments: 420

[2/8] MBtHiyY3FPo
Processing MBtHiyY3FPo...


  Audio: MBtHiyY3FPo.webm


...

  Audio: 85UTQ6OWDFU.webm


100%|██████████| 324656/324656 [04:30<00:00, 1200.34frames/s]


  Saved: t_07.json | segments: 994

[8/8] KrHmB8nCU3U
Processing KrHmB8nCU3U...


ERROR: [youtube] KrHmB8nCU3U: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


  ERROR: DownloadError: ERROR: [youtube] KrHmB8nCU3U: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

Done.
OK -> /content/raw/all_transcripts.json
Success: 7/8
Errors:
- {'video_id': 'KrHmB8nCU3U', 'error': 'DownloadError: ERROR: [youtube] KrHmB8nCU3U: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies'}


In [13]:
#download the lost video
import yt_dlp
from pathlib import Path

video_id = "51U3t948CvI"
url = f"https://www.youtube.com/watch?v={video_id}"

Path("audio").mkdir(exist_ok=True)

ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": f"audio/{video_id}.%(ext)s",
    "quiet": False,
    "noplaylist": True,
    "cookiefile": "/content/cookies.txt",
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

print("Download finished")

[youtube] Extracting URL: https://www.youtube.com/watch?v=51U3t948CvI
[youtube] 51U3t948CvI: Downloading webpage
[youtube] 51U3t948CvI: Downloading tv downgraded player API JSON
[youtube] 51U3t948CvI: Downloading web player API JSON
[youtube] 51U3t948CvI: Downloading web safari player API JSON


ERROR: [youtube] 51U3t948CvI: Requested format is not available. Use --list-formats for a list of available formats


DownloadError: ERROR: [youtube] 51U3t948CvI: Requested format is not available. Use --list-formats for a list of available formats

In [ ]:
#compress for downloading
!cd /content && zip -r raw_transcripts.zip raw

In [ ]:
#download zip
from google.colab import files
files.download("/content/raw_transcripts.zip")